# ML Pipeline Preparation
Follow the instructions below to help you create your ML pipeline.
### 1. Import libraries and load data from database.
- Import Python libraries
- Load dataset from database with [`read_sql_table`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_sql_table.html)
- Define feature and target variables X and Y

In [2]:
# import libraries
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# text processing
import re
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

# model persistence
import joblib

# ensure NLTK data present (only download if missing)
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('punkt')
    nltk.download('wordnet')

print('Libraries imported')

Libraries imported


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\roman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\roman\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
# load data from database
engine = create_engine('sqlite:///data/DisasterResponse.db')
df = pd.read_sql_table('disaster_messages', engine)

# define feature and target variables
X = df['message']
cols = list(df.columns)
Y = df[[c for c in cols if c not in ['id', 'message', 'original', 'genre']]]

print(f'Loaded data with shape: {df.shape}; X size: {X.shape}; Y shape: {Y.shape}')

Loaded data with shape: (26216, 40); X size: (26216,); Y shape: (26216, 36)


### 2. Write a tokenization function to process your text data

In [4]:
def tokenize(text):
    # normalize
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)
    # tokenize
    tokens = word_tokenize(text)
    # lemmatize
    lemmatizer = WordNetLemmatizer()
    clean_tokens = []
    for tok in tokens:
        clean_tok = lemmatizer.lemmatize(tok).lower().strip()
        clean_tokens.append(clean_tok)
    return clean_tokens

# quick test
print(tokenize('Emergency IN TOWN: need water and medical help!'))

['emergency', 'in', 'town', 'need', 'water', 'and', 'medical', 'help']


### 3. Build a machine learning pipeline
This machine pipeline should take in the `message` column as input and output classification results on the other 36 categories in the dataset. You may find the [MultiOutputClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.multioutput.MultiOutputClassifier.html) helpful for predicting multiple target variables.

In [5]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(tokenizer=tokenize, lowercase=True, stop_words='english')),
    ('clf', MultiOutputClassifier(RandomForestClassifier(n_jobs=-1, random_state=42)))
])

print('Pipeline defined')

Pipeline defined


### 4. Train pipeline
- Split data into train and test sets
- Train pipeline

In [6]:
# Split data and train pipeline
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(f"Training on {X_train.shape[0]} samples")

pipeline.fit(X_train, Y_train)
print('Training finished')

# Save the trained baseline pipeline (optional)
joblib.dump(pipeline, 'models/classifier_baseline.pkl')
print('Baseline model saved to models/classifier_baseline.pkl')

Training on 20972 samples


c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'u', 'wa'] not in stop_words.
  warnings.warn(


Training finished
Baseline model saved to models/classifier_baseline.pkl


### 5. Test your model
Report the f1 score, precision and recall for each output category of the dataset. You can do this by iterating through the columns and calling sklearn's `classification_report` on each.

In [7]:
# Evaluate model: per-category classification_report
import numpy as np
Y_pred = pipeline.predict(X_test)

for i, col in enumerate(Y_test.columns):
    print(f"Category: {col}")
    print(classification_report(Y_test[col], Y_pred[:, i], zero_division=0))
    print('-' * 60)

# Overall accuracy (average of label-wise accuracies)
accuracy = (Y_pred == Y_test.values).mean()
print(f'Overall label-wise accuracy (mean): {accuracy:.4f}')

Category: related
              precision    recall  f1-score   support

           0       0.69      0.43      0.53      1266
           1       0.83      0.94      0.88      3938
           2       0.78      0.17      0.29        40

    accuracy                           0.81      5244
   macro avg       0.77      0.51      0.56      5244
weighted avg       0.80      0.81      0.79      5244

------------------------------------------------------------
Category: request
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      4349
           1       0.82      0.47      0.60       895

    accuracy                           0.89      5244
   macro avg       0.86      0.72      0.77      5244
weighted avg       0.89      0.89      0.88      5244

------------------------------------------------------------
Category: offer
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5218
         

### 6. Improve your model
Use grid search to find better parameters. 

In [8]:
# Grid search to improve model (small example grid)
parameters = {
    'clf__estimator__n_estimators': [50, 100],
    'clf__estimator__min_samples_split': [2, 4]
}

cv = GridSearchCV(pipeline, param_grid=parameters, verbose=2, n_jobs=-1, cv=3)
print('GridSearchCV configured')

GridSearchCV configured


### 7. Test your model
Show the accuracy, precision, and recall of the tuned model.  

Since this project focuses on code quality, process, and  pipelines, there is no minimum performance metric needed to pass. However, make sure to fine tune your models for accuracy, precision and recall to make your project stand out - especially for your portfolio!

In [9]:
# Run grid search (optional; can be time-consuming)
# Uncomment to run
cv.fit(X_train, Y_train)
print('Grid search finished')
print(f'Best params: {cv.best_params_}')
best_model = cv.best_estimator_

print('Grid search cell ready — run manually if desired')

Fitting 3 folds for each of 4 candidates, totalling 12 fits


c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'u', 'wa'] not in stop_words.
  warnings.warn(


Grid search finished
Best params: {'clf__estimator__min_samples_split': 2, 'clf__estimator__n_estimators': 100}
Grid search cell ready — run manually if desired


### 8. Try improving your model further. Here are a few ideas:
* try other machine learning algorithms
* add other features besides the TF-IDF

In [9]:
# If a tuned model (cv) was trained, evaluate it here. Otherwise evaluate baseline pipeline saved earlier.
# Example with baseline pipeline:
model = pipeline
Y_pred = model.predict(X_test)

# For multilabel (MultiOutputClassifier), compute metrics per label
from sklearn.metrics import precision_score, recall_score, f1_score

# Compute per-label metrics and take the macro average
precisions = []
recalls = []
f1s = []

for i, col in enumerate(Y_test.columns):
    # Use average='macro' for multiclass support
    precision = precision_score(Y_test.iloc[:, i], Y_pred[:, i], average='macro', zero_division=0)
    recall = recall_score(Y_test.iloc[:, i], Y_pred[:, i], average='macro', zero_division=0)
    f1 = f1_score(Y_test.iloc[:, i], Y_pred[:, i], average='macro', zero_division=0)
    
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

print(f'Precision (macro avg across labels): {np.mean(precisions):.4f}')
print(f'Recall (macro avg across labels): {np.mean(recalls):.4f}')
print(f'F1 (macro avg across labels): {np.mean(f1s):.4f}')

# Subset accuracy: all labels must be correct for a sample
subset_accuracy = (Y_pred == Y_test.values).all(axis=1).mean()
print(f'Subset Accuracy (exact match): {subset_accuracy:.4f}')

Precision (macro avg across labels): 0.7778
Recall (macro avg across labels): 0.5990
F1 (macro avg across labels): 0.6172
Subset Accuracy (exact match): 0.2675


In [10]:
from sklearn.metrics import classification_report

for i, col in enumerate(Y_test.columns):
    print(f"Category: {col}")
    print(classification_report(Y_test[col], Y_pred[:, i]))



Category: related
              precision    recall  f1-score   support

           0       0.69      0.43      0.53      1266
           1       0.83      0.94      0.88      3938
           2       0.78      0.17      0.29        40

    accuracy                           0.81      5244
   macro avg       0.77      0.51      0.56      5244
weighted avg       0.80      0.81      0.79      5244

Category: request
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      4349
           1       0.82      0.47      0.60       895

    accuracy                           0.89      5244
   macro avg       0.86      0.72      0.77      5244
weighted avg       0.89      0.89      0.88      5244

Category: offer
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5218
           1       0.00      0.00      0.00        26

    accuracy                           1.00      5244
   macro avg       0.5

c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 

              precision    recall  f1-score   support

           0       0.97      1.00      0.99      5089
           1       0.55      0.07      0.13       155

    accuracy                           0.97      5244
   macro avg       0.76      0.53      0.56      5244
weighted avg       0.96      0.97      0.96      5244

Category: child_alone
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5244

    accuracy                           1.00      5244
   macro avg       1.00      1.00      1.00      5244
weighted avg       1.00      1.00      1.00      5244

Category: water
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      4905
           1       0.85      0.39      0.53       339

    accuracy                           0.96      5244
   macro avg       0.91      0.69      0.76      5244
weighted avg       0.95      0.96      0.95      5244

Category: food
              precis

c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\roman\OneDrive\Python\disaster_response_pipeline_project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 

### 9. Export your model as a pickle file

In [11]:
# Export the final model to a pickle file
# If you used GridSearchCV and want the tuned model, replace `pipeline` with `cv.best_estimator_` after fitting.
joblib.dump(pipeline, 'models/classifier.pkl')
print('Final model saved to models/classifier.pkl')

Final model saved to models/classifier.pkl


### 10. Use this notebook to complete `train_classifier.py`
Use the template file attached in the Resources folder to write a script that runs the steps above to create a database and export a model based on a new dataset specified by the user.